In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
PROJECT_ROOT = Path.cwd().parent

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
dataset = pd.read_csv(
    INTERIM_DIR /
    "earthquake_ground_motion_vs30.csv"
)

print("Shape:", dataset.shape)
print(
    "Events:",
    dataset["event_id"].nunique()
)

display(dataset.head())

Shape: (2912739, 18)
Events: 40


,grid_lon,grid_lat,PGA,PGV,MMI,PSA03,PSA10,PSA30,event_id,magnitude,event_lat,event_lon,epicentral_distance_km,hypocentral_distance_km,event_depth,Vs30,Vs30_clean,site_class
0,69.3004,38.2842,0.40,0.17,2.54,0.71,0.18,0.03,us10002mft,5.5,36.68,71.3,250.885571,315.316619,191.0,603.00000,NaN,Unknown
1,69.3171,38.2842,0.41,0.18,2.57,0.74,0.19,0.03,us10002mft,5.5,36.68,71.3,249.851709,314.494636,191.0,523.98706,523.98706,Stiff Soil / Rock
2,69.3337,38.2842,0.40,0.16,2.53,0.71,0.17,0.03,us10002mft,5.5,36.68,71.3,248.828424,313.682299,191.0,603.00000,NaN,Unknown
3,69.3504,38.2842,0.38,0.15,2.46,0.68,0.16,0.03,us10002mft,5.5,36.68,71.3,247.803445,312.869857,191.0,649.13720,649.13720,Stiff Soil / Rock
4,69.3671,38.2842,0.40,0.16,2.50,0.71,0.17,0.03,us10002mft,5.5,36.68,71.3,246.783004,312.062255,191.0,574.54755,574.54755,Stiff Soil / Rock


In [4]:
print(
    "PGA minimum:",
    dataset["PGA"].min()
)

print(
    "PGA maximum:",
    dataset["PGA"].max()
)

print(
    "Zero PGA:",
    (dataset["PGA"] == 0).sum()
)

print(
    "Missing PGA:",
    dataset["PGA"].isna().sum()
)

PGA minimum: 0.0
PGA maximum: 90.17
Zero PGA: 39370
Missing PGA: 0


In [5]:
ml_data = dataset[
    dataset["PGA"] > 0
].copy()

print(
    "Rows after PGA filtering:",
    f"{len(ml_data):,}"
)

Rows after PGA filtering: 2,873,369


In [6]:
ml_data["log_PGA"] = np.log10(
    ml_data["PGA"]
)

display(
    ml_data[
        [
            "PGA",
            "log_PGA"
        ]
    ].describe()
)

,PGA,log_PGA
count,2.873369e+06,2.873369e+06
mean,6.049253e-01,-6.032711e-01
std,1.246439e+00,6.247126e-01
min,1.000000e-02,-2.000000e+00
25%,1.000000e-01,-1.000000e+00
50%,2.900000e-01,-5.376020e-01
75%,7.000000e-01,-1.549020e-01
max,9.017000e+01,1.955062e+00


In [7]:
ml_data["log_distance"] = np.log10(
    ml_data["hypocentral_distance_km"].clip(
        lower=1
    )
)

ml_data["log_epicentral_distance"] = np.log10(
    ml_data["epicentral_distance_km"].clip(
        lower=1
    )
)

In [8]:
ml_data["magnitude_distance_interaction"] = (
    ml_data["magnitude"]
    * ml_data["log_distance"]
)

In [9]:
ml_data["magnitude_squared"] = (
    ml_data["magnitude"] ** 2
)

In [10]:
ml_data["depth_distance_interaction"] = (
    ml_data["event_depth"]
    * ml_data["log_distance"]
)

In [11]:
ml_data["log_Vs30"] = np.log10(
    ml_data["Vs30_clean"]
)

In [12]:
print(
    "Missing log Vs30:",
    ml_data["log_Vs30"].isna().sum()
)

Missing log Vs30: 2001676


In [13]:
ml_data["Vs30_available"] = (
    ml_data["Vs30_clean"].notna()
).astype(int)

In [14]:
feature_columns = [
    "magnitude",
    "event_depth",

    "epicentral_distance_km",
    "hypocentral_distance_km",

    "log_distance",

    "magnitude_squared",
    "magnitude_distance_interaction",
    "depth_distance_interaction",

    "Vs30_clean",
    "log_Vs30",
    "Vs30_available",

    "grid_lat",
    "grid_lon"
]

target_column = "log_PGA"

In [30]:
ml_data = ml_data[
    ml_data["Vs30_clean"].notna()
].copy()

print(
    "Final terrestrial ML observations:",
    f"{len(ml_data):,}"
)

print(
    "Events:",
    ml_data["event_id"].nunique()
)

print(
    "Missing Vs30:",
    ml_data["Vs30_clean"].isna().sum()
)

Final terrestrial ML observations: 871,693
Events: 40
Missing Vs30: 0


In [31]:
event_counts = (
    ml_data
    .groupby("event_id")
    .size()
    .sort_values()
)

display(
    event_counts
)

print(
    "\nMinimum observations/event:",
    event_counts.min()
)

print(
    "Maximum observations/event:",
    event_counts.max()
)

event_id
b000sg5e         10
b000ryuh         22
b000l2rk         25
us10002n5q       96
b000l4ju        359
c000nb99        642
us100088sf      760
c000njsg        810
usc000syca      949
us2000cp4g     2074
c000kmdj       2174
b000ry7u       2363
us1000ggp5     2669
us10003vsp     2750
us10003vpz     3172
us2000d1a1     4571
c000njrq       5204
us10003vxc     5608
us10007tps     5619
us20002z57     6289
b000qy82       7826
us10003vry    11175
us20008k1z    14503
us10002mft    42706
b000g112      43149
us100030qs    43388
us10004dtm    43768
us10004rhs    43996
c000rff2      44291
us100042n2    44345
b000hdu8      44781
us20008hyg    45531
us200082s5    45990
usb000syze    46662
us2000299v    47846
usc000sy0y    48778
b000fzn7      51108
b000sfrw      51465
b000ktbm      51526
c000ff4h      52693
dtype: int64


Minimum observations/event: 10
Maximum observations/event: 52693


In [32]:
display(
    ml_data[
        [
            "magnitude",
            "event_depth",
            "epicentral_distance_km",
            "hypocentral_distance_km",
            "Vs30_clean",
            "PGA",
            "log_PGA"
        ]
    ].describe()
)

,magnitude,event_depth,epicentral_distance_km,hypocentral_distance_km,Vs30_clean,PGA,log_PGA
count,871693.000000,871693.000000,871693.000000,871693.000000,871693.000000,871693.000000,871693.000000
mean,5.433896,84.476340,152.780404,191.501440,598.394522,0.674965,-0.393291
std,0.299119,83.426799,67.241685,72.710181,265.239844,0.719785,0.469840
min,5.100000,6.610000,0.111195,10.006840,180.000000,0.010000,-2.000000
25%,5.200000,13.610000,108.398365,140.275266,341.877870,0.180000,-0.744727
50%,5.400000,38.700000,153.321748,194.041749,592.079960,0.460000,-0.337242
75%,5.600000,191.000000,189.845794,242.395564,900.000000,0.890000,-0.050610
max,6.600000,239.000000,556.895624,556.985401,900.000000,16.560000,1.219060


In [33]:
events = (
    ml_data["event_id"]
    .drop_duplicates()
    .tolist()
)

print(
    "Total events:",
    len(events)
)

Total events: 40


In [34]:
rng = np.random.default_rng(42)

events = np.array(events)

rng.shuffle(events)

n_events = len(events)

n_train = int(
    0.70 * n_events
)

n_val = int(
    0.15 * n_events
)

train_events = events[
    :n_train
]

val_events = events[
    n_train:n_train + n_val
]

test_events = events[
    n_train + n_val:
]

print(
    "Train events:",
    len(train_events)
)

print(
    "Validation events:",
    len(val_events)
)

print(
    "Test events:",
    len(test_events)
)

Train events: 28
Validation events: 6
Test events: 6


In [35]:
train_data = ml_data[
    ml_data["event_id"].isin(
        train_events
    )
].copy()

In [36]:
val_data = ml_data[
    ml_data["event_id"].isin(
        val_events
    )
].copy()

In [37]:
test_data = ml_data[
    ml_data["event_id"].isin(
        test_events
    )
].copy()

In [38]:
print(
    "Train ∩ Validation:",
    len(
        set(train_events)
        &
        set(val_events)
    )
)

print(
    "Train ∩ Test:",
    len(
        set(train_events)
        &
        set(test_events)
    )
)

print(
    "Validation ∩ Test:",
    len(
        set(val_events)
        &
        set(test_events)
    )
)

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [39]:
print(
    "Training rows:",
    f"{len(train_data):,}"
)

print(
    "Validation rows:",
    f"{len(val_data):,}"
)

print(
    "Test rows:",
    f"{len(test_data):,}"
)

Training rows: 572,844
Validation rows: 112,799
Test rows: 186,050


In [40]:
train_correlations = (
    train_data[
        feature_columns + [target_column]
    ]
    .corr()[target_column]
    .sort_values(
        ascending=False
    )
)

display(
    train_correlations
)

log_PGA                           1.000000
event_depth                       0.525702
depth_distance_interaction        0.498285
magnitude                         0.388698
magnitude_squared                 0.370809
grid_lat                          0.323821
Vs30_clean                        0.127857
log_Vs30                          0.118923
magnitude_distance_interaction   -0.022705
hypocentral_distance_km          -0.243283
log_distance                     -0.297400
grid_lon                         -0.534492
epicentral_distance_km           -0.574514
Vs30_available                         NaN
Name: log_PGA, dtype: float64

In [41]:
event_split = pd.DataFrame({
    "event_id": np.concatenate([
        train_events,
        val_events,
        test_events
    ]),
    "split": (
        ["train"] * len(train_events)
        +
        ["validation"] * len(val_events)
        +
        ["test"] * len(test_events)
    )
})

display(
    event_split
)

,event_id,split
0,c000ff4h,train
1,us100042n2,train
2,usc000syca,train
3,us2000cp4g,train
4,us10003vry,train
5,c000rff2,train
6,us20008hyg,train
7,us10003vsp,train
8,b000l4ju,train
9,b000ryuh,train


In [25]:
event_split.to_csv(
    PROCESSED_DIR /
    "event_split.csv",
    index=False
)

In [26]:
train_data.to_csv(
    PROCESSED_DIR /
    "train_data.csv",
    index=False
)

val_data.to_csv(
    PROCESSED_DIR /
    "validation_data.csv",
    index=False
)

test_data.to_csv(
    PROCESSED_DIR /
    "test_data.csv",
    index=False
)

print("ML datasets saved.")

ML datasets saved.


In [27]:
summary = pd.DataFrame({
    "Dataset": [
        "Train",
        "Validation",
        "Test"
    ],
    "Events": [
        train_data["event_id"].nunique(),
        val_data["event_id"].nunique(),
        test_data["event_id"].nunique()
    ],
    "Rows": [
        len(train_data),
        len(val_data),
        len(test_data)
    ]
})

display(summary)

,Dataset,Events,Rows
0,Train,28,2092176
1,Validation,6,468857
2,Test,6,312336


In [42]:
feature_config = pd.DataFrame({
    "feature": feature_columns,
    "target": [
        "YES" if x == target_column else "NO"
        for x in feature_columns
    ]
})

feature_config.to_csv(
    PROCESSED_DIR /
    "feature_config.csv",
    index=False
)

display(feature_config)

,feature,target
0,magnitude,NO
1,event_depth,NO
2,epicentral_distance_km,NO
3,hypocentral_distance_km,NO
4,log_distance,NO
5,magnitude_squared,NO
6,magnitude_distance_interaction,NO
7,depth_distance_interaction,NO
8,Vs30_clean,NO
9,log_Vs30,NO


In [43]:
print("Total observations:", len(ml_data))

print("\nVs30 special/raw values:")

for value, label in {
    600: "Ocean",
    601: "Ice",
    602: "Glacier",
    603: "Lake"
}.items():

    count = (
        ml_data["Vs30"]
        .eq(value)
        .sum()
    )

    print(
        f"{label:10s}: "
        f"{count:,} "
        f"({count / len(ml_data) * 100:.2f}%)"
    )

print("\nValid Vs30:")
print(
    ml_data["Vs30_clean"].notna().sum(),
    f"({ml_data['Vs30_clean'].notna().mean()*100:.2f}%)"
)

Total observations: 871693

Vs30 special/raw values:
Ocean     : 0 (0.00%)
Ice       : 0 (0.00%)
Glacier   : 0 (0.00%)
Lake      : 0 (0.00%)

Valid Vs30:
871693 (100.00%)
